# 5. Evaluation: From Gut Checks to Automated Judgment

### Setup

In [ ]:
import sys 
sys.path.insert(0, "..") 
from config import API_KEY as key, ENDPOINT_BASE as endpoint_base 
import requests 
import json 
import chromadb
from datetime import datetime

# connect to the chromadb
client = chromadb.PersistentClient(path="../chroma_db")
collection = client.get_collection(name="rpg_rules")

from sentence_transformers import SentenceTransformer
embed_model = SentenceTransformer("../models/granite-embedding-30m-english")


print(f"✅ Connected to ChromaDB — {collection.count()} chunks loaded")


## 5.1 The Manual Phase: Ask Questions, Write Down What's Wrong

This is how every customer starts. Someone sits in front of the system, asks a few questions, and forms an opinion. That opinion usually sounds like one of these:

* "It kind of works."
* "Some answers are good, some are weird."
* "I don't trust it yet."

We're going to do the same thing, deliberately. But unlike the customer, we're going to be structured about it. We already have two questions from earlier in the lab. Now we're going to expand that to a small set, run each question two ways, without context and with RAG,  and record what we observe.

This isn't sophisticated. It's not meant to be. The point is to do what the customer does, feel why it breaks down, and understand what we'd need to do it better.


In [6]:
eval_questions = [
    {
        "id": "q1",
        "question": "What happens if a Thief fails an Open Locks attempt?",
        "known_answer": "The Thief may only try once per lock. If they fail, they must wait until they gain another level of experience before trying again.",
        "why": "Control question — we already verified this in Section 4."
    },
    {
        "id": "q2",
        "question": "Why can't Elves roll higher than a d6 for hit points?",
        "known_answer": "Elves use a d6 for hit dice because of their character class restrictions in Basic Fantasy RPG.",
        "why": "Tests whether the model answers from this game or defaults to D&D."
    },
    {
        "id": "q3",
        "question": "Can a Fighter use magic-user scrolls?",
        "known_answer": "No. Only Magic-Users (and in some cases Thieves at higher levels) can use magic-user scrolls.",
        "why": "Requires interpreting class restriction rules — not a single sentence answer."
    },
    {
        "id": "q4",
        "question": "How is initiative determined in combat?",
        "known_answer": "Each side rolls 1d6 at the start of each round. The side with the higher roll acts first.",
        "why": "A rule that differs significantly across RPG systems. Tests domain specificity."
    },
    {
        "id": "q5",
        "question": "What is the maximum number of retainers a character can hire?",
        "known_answer": "This depends on the character's Charisma score. The Charisma table defines the maximum number of retainers.",
        "why": "Answer requires connecting two pieces of information — Charisma rules and retainer rules."
    },
]

print(f"Evaluation set: {len(eval_questions)} questions loaded")
for q in eval_questions:
    print(f"  [{q['id']}] {q['question']}")

Evaluation set: 5 questions loaded
  [q1] What happens if a Thief fails an Open Locks attempt?
  [q2] Why can't Elves roll higher than a d6 for hit points?
  [q3] Can a Fighter use magic-user scrolls?
  [q4] How is initiative determined in combat?
  [q5] What is the maximum number of retainers a character can hire?


In [8]:
url_chat = f"{endpoint_base}/chat/completions"
model_id = "granite-3-2-8b-instruct"

def ask_without_context(question, model=model_id):
    """Baseline: model answers from its own training data only."""
    headers = {
        "Authorization": f"Bearer {key}",
        "Content-Type": "application/json"
    }
    data = {
        "model": model,
        "messages": [{"role": "user", "content": question}],
        "max_tokens": 300,
        "temperature": 0
    }
    try:
        response = requests.post(url_chat, headers=headers, json=data)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]
    except Exception as e:
        return f"ERROR: {e}"


def ask_with_rag(question, collection, model=model_id, n_results=3):
    """RAG: retrieve relevant chunks, then ask the model."""
    # Retrieve (embed the question manually)
    query_embedding = embed_model.encode(question).tolist()
    results = collection.query(query_embeddings=[query_embedding], n_results=n_results)
    context_chunks = results["documents"][0]
    context = "\n\n---\n\n".join(context_chunks)

    # Build prompt
    system_msg = (
        "Answer the question using only the provided context. "
        "If the context does not contain enough information, say so."
    )
    headers = {
        "Authorization": f"Bearer {key}",
        "Content-Type": "application/json"
    }
    data = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ],
        "max_tokens": 300,
        "temperature": 0
    }
    try:
        response = requests.post(url_chat, headers=headers, json=data)
        response.raise_for_status()
        answer = response.json()["choices"][0]["message"]["content"]
        return answer, context_chunks
    except Exception as e:
        return f"ERROR: {e}", context_chunks

In [11]:
results = []

print("Running evaluation set...\n")
for q in eval_questions:
    print(f"[{q['id']}] {q['question']}")

    # Baseline (no context)
    baseline_answer = ask_without_context(q["question"])

    # RAG (with retrieved context)
    rag_answer, retrieved_chunks = ask_with_rag(q["question"], collection)

    result = {
        "id": q["id"],
        "question": q["question"],
        "known_answer": q["known_answer"],
        "baseline_answer": baseline_answer,
        "rag_answer": rag_answer,
        "retrieved_chunks": retrieved_chunks,
        "timestamp": datetime.now().isoformat()
    }
    results.append(result)
    print(f"  ✅ Done\n")

print(f"All {len(results)} questions complete.")


Running evaluation set...

[q1] What happens if a Thief fails an Open Locks attempt?
  ✅ Done

[q2] Why can't Elves roll higher than a d6 for hit points?
  ✅ Done

[q3] Can a Fighter use magic-user scrolls?
  ✅ Done

[q4] How is initiative determined in combat?
  ✅ Done

[q5] What is the maximum number of retainers a character can hire?
  ✅ Done

All 5 questions complete.


In [ ]:
print("=" * 80)
print("EVALUATION RESULTS: Baseline vs. RAG")
print("=" * 80)

for r in results:
    print(f"\n{'─' * 80}")
    print(f"{r['id'].upper()}: {r['question']}")
    print(f"{'─' * 80}")
    print(f"\n🧠 Baseline (no context):\n{r['baseline_answer']}")
    print(f"\n📚 RAG (with retrieved context):\n{r['rag_answer']}")
    print(f"\n📎 Retrieved chunks: {len(r['retrieved_chunks'])}")

In [ ]:
# Simple manual scoring framework
print("SCORING GUIDE")
print("  0 = Wrong or hallucinated")
print("  1 = Partially correct or vague")
print("  2 = Correct and grounded\n")

for r in results:
    print(f"{r['id'].upper()}: {r['question']}")
    print(f"  Baseline: ___")
    print(f"  RAG:      ___")
    print()

In [ ]:
def show_scores():
    print("EVALUATION SUMMARY")
    print("=" * 50)
    print(f"{'Question':<12} {'Baseline':>10} {'RAG':>10} {'Delta':>10}")
    print("-" * 50)
    
    baseline_total = 0
    rag_total = 0
    
    for q_id, s in sorted(scores.items()):
        b = s.get("baseline", "—")
        r = s.get("rag", "—")
        delta = ""
        if isinstance(b, int) and isinstance(r, int):
            delta = f"+{r - b}" if r >= b else str(r - b)
            baseline_total += b
            rag_total += r
        print(f"{q_id.upper():<12} {str(b):>10} {str(r):>10} {delta:>10}")
    
    print("-" * 50)
    print(f"{'Total':<12} {baseline_total:>10} {rag_total:>10} {f'+{rag_total - baseline_total}':>10}")

show_scores()

## 5.3  From Manual Checks to Structured Evaluation: Add expected answers so you can score programmatically instead of just eyeballing:


In [20]:
eval_set = [
    {
        "id": "q1",
        "question": "What happens if a Thief fails an Open Locks attempt?",
        "expected_keywords": ["wait", "another level", "experience", "once", "lock"],
        "reference_answer": "The Thief must wait until they have gained another level of experience before trying again. It may only be tried once per lock."
    },
    {
        "id": "q2",
        "question": "Why can't Elves roll higher than a d6 for hit points?",
        "expected_keywords": ["d6", "hit die", "elf", "elves", "magic-user", "combination"],
        "reference_answer": "Elves use a d6 for hit dice because they are a combination Fighter/Magic-User class, and their hit die reflects the Magic-User side."
    },
    {
        "id": "q3",
        "question": "Can a Fighter use magic-user scrolls?",
        "expected_keywords": ["no", "cannot", "magic-user", "arcane", "spell caster"],
        "reference_answer": "No. Only spell casters can use magic-user scrolls."
    },
    {
        "id": "q4",
        "question": "How is initiative determined in combat?",
        "expected_keywords": ["d6", "each side", "roll", "beginning", "round"],
        "reference_answer": "Each side rolls 1d6 at the beginning of each round. The side with the highest roll acts first."
    },
    {
        "id": "q5",
        "question": "What is the maximum number of retainers a character can hire?",
        "expected_keywords": ["charisma", "bonus", "4", "retainer"],
        "reference_answer": "The number of retainers is based on the character's Charisma bonus. A character with average Charisma can hire up to 4."
    },
]

print(f"✅ Evaluation set defined — {len(eval_set)} questions with expected answers")

✅ Evaluation set defined — 5 questions with expected answers


In [21]:
def keyword_score(answer, expected_keywords):
    """Score based on how many expected keywords appear in the answer."""
    answer_lower = answer.lower()
    hits = [kw for kw in expected_keywords if kw.lower() in answer_lower]
    misses = [kw for kw in expected_keywords if kw.lower() not in answer_lower]
    score = len(hits) / len(expected_keywords) if expected_keywords else 0
    return {
        "score": round(score, 2),
        "hits": hits,
        "misses": misses
    }

def run_structured_eval(eval_set, results):
    """Run keyword scoring against both baseline and RAG answers."""
    scored = []
    
    for ev in eval_set:
        # Find the matching result
        match = next((r for r in results if r["id"] == ev["id"]), None)
        if not match:
            print(f"⚠️  No result found for {ev['id']}, skipping")
            continue
        
        baseline_eval = keyword_score(match["baseline_answer"], ev["expected_keywords"])
        rag_eval = keyword_score(match["rag_answer"], ev["expected_keywords"])
        
        scored.append({
            "id": ev["id"],
            "question": ev["question"],
            "reference": ev["reference_answer"],
            "baseline_answer": match["baseline_answer"],
            "rag_answer": match["rag_answer"],
            "baseline_score": baseline_eval,
            "rag_score": rag_eval,
        })
    
    return scored

scored_results = run_structured_eval(eval_set, results)
print(f"✅ Structured evaluation complete — {len(scored_results)} questions scored")

✅ Structured evaluation complete — 5 questions scored


In [ ]:
display(HTML("""
<div style="background:rgba(255,152,0,0.1); padding:12px; border-radius:8px; border-left:4px solid #FF9800;">
<h3>5.3 — Structured Evaluation: Keyword Coverage</h3>
<p>Instead of eyeballing, we now check whether answers contain the key terms 
we'd expect from a correct response. This isn't perfect — but it's <strong>repeatable</strong> and <strong>defensible</strong>.</p>
</div>
"""))

for s in scored_results:
    b = s["baseline_score"]
    r = s["rag_score"]
    delta = r["score"] - b["score"]
    indicator = "📈" if delta > 0 else "📉" if delta < 0 else "➡️"
    
    print(f"\n{'═' * 70}")
    print(f"{s['id'].upper()}: {s['question']}")
    print(f"{'═' * 70}")
    
    print(f"\n📖 Reference answer:")
    print(f"   {s['reference']}")
    
    print(f"\n🧠 Baseline — score: {b['score']:.0%}")
    print(f"   ✓ hits:   {', '.join(b['hits']) or '(none)'}")
    print(f"   ✗ misses: {', '.join(b['misses']) or '(none)'}")
    
    print(f"\n📚 RAG — score: {r['score']:.0%}")
    print(f"   ✓ hits:   {', '.join(r['hits']) or '(none)'}")
    print(f"   ✗ misses: {', '.join(r['misses']) or '(none)'}")
    
    print(f"\n{indicator} Delta: {delta:+.0%}")

In [ ]:
print("STRUCTURED EVALUATION SUMMARY")
print("=" * 55)
print(f"{'Question':<10} {'Baseline':>12} {'RAG':>12} {'Delta':>12}")
print("-" * 55)

b_total = 0
r_total = 0

for s in scored_results:
    b = s["baseline_score"]["score"]
    r = s["rag_score"]["score"]
    delta = r - b
    b_total += b
    r_total += r
    indicator = "📈" if delta > 0 else "📉" if delta < 0 else "➡️"
    print(f"{s['id'].upper():<10} {b:>11.0%} {r:>11.0%} {delta:>+11.0%} {indicator}")

n = len(scored_results)
print("-" * 55)
print(f"{'Average':<10} {b_total/n:>11.0%} {r_total/n:>11.0%} {(r_total-b_total)/n:>+11.0%}")

print(f"\n{'─' * 55}")
print("Key takeaway:")
if r_total > b_total:
    print("  RAG improves keyword coverage — retrieval is adding real value.")
    print("  Next: Are the remaining misses a retrieval problem or a model problem?")
elif r_total == b_total:
    print("  No measurable improvement. Check if retrieval is finding the right chunks.")
else:
    print("  RAG is underperforming baseline. Retrieved context may be introducing noise.")

In [24]:
display(HTML("""
<div style="background:rgba(33,150,243,0.1); padding:15px; border-radius:8px; border-left:4px solid #2196F3;">
<h3>🔍 What Structured Evaluation Reveals</h3>
<p>Keyword scoring is simple and imperfect — but it gives us something critical: <strong>a repeatable signal</strong>.</p>
<p>Notice what we can now do that we couldn't before:</p>
<ol>
    <li><strong>Compare iterations:</strong> Change the chunking strategy, re-run, and see if scores improve</li>
    <li><strong>Locate failures:</strong> Missing keywords tell us <em>what</em> the answer lacked — then we check <em>why</em></li>
    <li><strong>Defend decisions:</strong> "RAG improved keyword coverage from 40% to 80%" is a statement you can bring to a customer</li>
</ol>
<p><strong>What this doesn't tell us:</strong> Whether the answer is <em>coherent</em>, <em>well-phrased</em>, or <em>correctly reasoned</em>. 
That still requires human judgment — which is why Section 5.2 exists alongside this one.</p>
<h4>Next: The Escalation Ladder</h4>
<p>We now have two kinds of evidence: human scores and automated coverage. 
The question becomes: <strong>where is the system failing, and what's the right fix?</strong></p>
</div>
"""))